In [ ]:
# imports
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.utils.multiclass import unique_labels
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
plt.style.use('classic')

In [ ]:
# load data
df = pd.read_csv("../data/raw/train.csv")
df_test = pd.read_csv("../data/raw/test.csv")
df.head()

In [ ]:
df_test.head()

In [ ]:
categorical_features = [ 'gender', 'Partner', 'Dependents',  'PhoneService', 'MultipleLines', 'InternetService', 'DeviceProtection', 'OnlineSecurity', 'OnlineBackup', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'PaymentMethod']
ordinal_features = ['Contract']
metric_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

In [ ]:
# Histogram for ordinal features
plt.figure(figsize=(6,4))
sns.histplot(data=df, x=df['Contract'])
plt.title('Histogram Contract')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram metric features
for column in metric_features:
    plt.figure(figsize=(6,4))
    sns.histplot(data=df, x=column)
    plt.title(f"Histogram of {column}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Count plot
for column in categorical_features:
    plt.figure(figsize=(6,4))
    sns.countplot(data=df, x=column, hue='Churn')
    plt.title(f"{column} vs Churn")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Boxplot
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
sns.countplot(x='Churn', data=df, ax=axes[0,0])
axes[0,0].set_title('Churn Distribution')
sns.boxplot(x='Churn', y='MonthlyCharges', data=df, ax=axes[0,1])
axes[0,1].set_title('MonthlyCharges vs Churn')
sns.boxplot(x='Churn', y='tenure', data=df, ax=axes[1,0])
axes[1,0].set_title('Tenure vs Churn')
sns.boxplot(x='Churn', y='TotalCharges', data=df, ax=axes[1,1])
axes[1,1].set_title('TotalCharges vs Churn')
plt.tight_layout()
plt.show()

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

In [ ]:
# convert strings to numeric values
# get unique values from each categorical feature
cat_unique_labels = [unique_labels(X[column]) for column in categorical_features]
for i, _ in enumerate(categorical_features):
    print(i)
    print(categorical_features[i], ' :', cat_unique_labels[i])

# get unique values from contract
contract_unique_labels = unique_labels(X['Contract'])
print(contract_unique_labels)

In [ ]:
# Create a ColumnTransformer to apply transformations only to specific columns.
# 'remainder="passthrough"' means other columns will be left unchanged.
column_transformer = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(categories=cat_unique_labels), categorical_features),
        ('ordinal', OrdinalEncoder(categories=[contract_unique_labels]), ordinal_features)
    ],
    remainder='passthrough'
)
# Fit and transform the features
X_transformed = column_transformer.fit_transform(X)

# Transform the target using LabelEncoder (or another appropriate encoder)
target_encoder = LabelEncoder()
y_transformed = target_encoder.fit_transform(y)

# Now X_transformed and y_transformed are ready for model training
print(pd.DataFrame(X_transformed, columns=column_transformer.get_feature_names_out()))
print(y_transformed)

In [ ]:
X_transformed_pd = pd.DataFrame(X_transformed)
X_transformed_pd.head()